In [3]:
import os
import io
from copy import deepcopy
from docx import Document
from docx.oxml.ns import qn
from docx.opc.constants import RELATIONSHIP_TYPE as RT

# ==== НАСТРОЙКИ ====
folder = r"/home/tishchenkova@lofi.pgt/Документы/Договора/ГЭХ/2026.11.17"
output_name = "итог.docx"
# ====================

output_path = os.path.join(folder, output_name)

files = sorted(
    f for f in os.listdir(folder)
    if f.lower().endswith(".docx")
    and not f.startswith("~$")
    and f != output_name
)
if not files:
    raise SystemExit("Нет .docx-файлов")

target = Document()
body = target.element.body
for p in list(body.findall(qn("w:p"))):
    body.remove(p)
sectPr = body.find(qn("w:sectPr"))


def import_images(src_doc, tgt_doc, element):
    """Добавить все картинки из src_doc в tgt_doc и переназначить rId в element."""
    # 1) карта старый rId -> новый rId (для всех IMAGE-связей исходника)
    rid_map = {}
    for rid, rel in src_doc.part.rels.items():
        if rel.reltype == RT.IMAGE and hasattr(rel.target_part, "blob"):
            new_rid, _ = tgt_doc.part.get_or_add_image(
                io.BytesIO(rel.target_part.blob)
            )
            rid_map[rid] = new_rid

    if not rid_map:
        return

    # 2) подменяем ссылки внутри скопированного элемента
    for blip in element.iter(qn("a:blip")):
        for attr in ("r:embed", "r:link"):
            old = blip.get(qn(attr))
            if old and old in rid_map:
                blip.set(qn(attr), rid_map[old])


for i, filename in enumerate(files):
    if i > 0:
        target.add_page_break()

    src = Document(os.path.join(folder, filename))
    for el in src.element.body:
        if el.tag == qn("w:sectPr"):
            continue
        new_el = deepcopy(el)
        import_images(src, target, new_el)   # <-- вот это и чинит картинки
        sectPr.addprevious(new_el)

target.save(output_path)
print(f"Готово: {output_path}")
print(f"Объединено файлов: {len(files)}")

Готово: /home/tishchenkova@lofi.pgt/Документы/Договора/ГЭХ/2026.11.17/итог.docx
Объединено файлов: 31


In [10]:
import os
import sys
import json
import shutil
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font



# ================== НАСТРОЙКИ ==================
ROOT         = r"/home/tishchenkova@lofi.pgt/Документы/Договора/ГЭХ/2026.11.23"
EXCLUDE_FILE = os.path.join(ROOT, "Исключить.xlsx")
MODELS_XLSX  = os.path.join(ROOT, "Модели.xlsx")
PARAMS_XLSX  = os.path.join(ROOT, "Параметры.xlsx")
MODELS_DIR   = os.path.join(ROOT, "models")
CONFIGS_DIR  = os.path.join(ROOT, "configs")
NN_DIR       = os.path.join(ROOT, "nn_models")
# ===============================================

def format_sheet(ws):
    """Форматирование по названиям столбцов (строка 1 — заголовки)."""
    CENTER = Alignment(horizontal="center", vertical="center")
    TEXT   = Alignment(horizontal="left",   vertical="top",    wrap_text=True)
    CODE   = Alignment(horizontal="left",   vertical="top",    wrap_text=True)
    OTHER  = Alignment(horizontal="left",   vertical="center", wrap_text=False)

    last = find_last_data_row(ws)

    for c in range(1, ws.max_column + 1):
        h_cell = ws.cell(row=1, column=c)
        h = str(h_cell.value).strip().lower() if h_cell.value is not None else ""

        # заголовок: полужирный, по центру
        h_cell.font = Font(name=h_cell.font.name, size=h_cell.font.size, bold=True,
                           color=h_cell.font.color)
        h_cell.alignment = CENTER

        # выбор выравнивания для значений столбца
        if h == "#":
            align = CENTER
        elif "kks" in h:
            align = CENTER
        elif "описание" in h:
            align = TEXT
        elif "тип" in h and "модели" in h:
            align = TEXT
        elif "ед" in h and "изм" in h:
            align = CENTER
        elif h == "код":
            align = CODE
        else:
            align = OTHER

        for r in range(2, last + 1):
            ws.cell(row=r, column=c).alignment = align


def log(msg):
    print(msg)


def find_last_data_row(ws):
    for r in range(ws.max_row, 0, -1):
        for c in range(1, ws.max_column + 1):
            v = ws.cell(row=r, column=c).value
            if v is not None and str(v).strip() != "":
                return r
    return 1


# ---------- 1. Список исключений ----------
def load_exclude_list(path):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Нет файла исключений: {path}")
    wb = load_workbook(path, data_only=True)
    ws = wb.active
    start = 1
    first = ws.cell(row=1, column=1).value
    if first and "kks" in str(first).lower():
        start = 2
    result = set()
    for r in range(start, ws.max_row + 1):
        v = ws.cell(row=r, column=1).value
        if v is not None and str(v).strip():
            result.add(str(v).strip())
    return result


# ---------- Утилита: добавить столбец "#" первым ----------
from openpyxl.styles import Alignment

def add_index_column(ws, header_val="#", ref_col=None):
    """
    Вставляет столбец "#" первым и нумерует позиции.
    ref_col — номер эталонного столбца (1-based, ДО вставки), например KKS модели.
    Один номер = один блок эталонного столбца (объединённый диапазон или одна строка);
    ячейка # объединяется ровно по этому диапазону.
    Если ref_col=None — просто нумерует строки подряд, без объединений.
    """
    old_merges = [(mr.min_col, mr.min_row, mr.max_col, mr.max_row)
                  for mr in ws.merged_cells.ranges]

    # снимаем объединения, вставляем столбец
    for mr in list(ws.merged_cells.ranges):
        ws.unmerge_cells(str(mr))
    ws.insert_cols(1)
    ws.cell(row=1, column=1, value=header_val)

    # восстанавливаем объединения со сдвигом столбцов +1
    new_merges = []
    for (c1, r1, c2, r2) in old_merges:
        new_merges.append((c1 + 1, r1, c2 + 1, r2))
        try:
            ws.merge_cells(start_row=r1, start_column=c1 + 1,
                           end_row=r2, end_column=c2 + 1)
        except Exception:
            pass

    last = find_last_data_row(ws)

    # блоки эталонного столбца: {первая_строка: последняя_строка}
    spans = {}
    if ref_col:
        rc = ref_col + 1  # после вставки столбец сдвинулся
        for (c1, r1, c2, r2) in new_merges:
            if c1 <= rc <= c2 and r2 > r1 and r1 >= 2:
                spans[r1] = r2

    center = Alignment(horizontal="center", vertical="center")
    num = 0
    r = 2
    while r <= last:
        end = spans.get(r, r)
        num += 1
        cell = ws.cell(row=r, column=1, value=num)
        cell.alignment = center
        if end > r:
            try:
                ws.merge_cells(start_row=r, start_column=1,
                               end_row=end, end_column=1)
            except Exception:
                pass
        r = end + 1


# ---------- 2. Модели.xlsx ----------
def update_models(path, exclude):
    wb = load_workbook(path)

    ws = None
    for name in wb.sheetnames:
        header = [c.value for c in wb[name][1]]
        if any(h and "kks" in str(h).lower() and "модели" in str(h).lower()
               for h in header):
            ws = wb[name]
            break
    if ws is None:
        ws = wb.active

    header = [c.value for c in ws[1]]
    kks_col = None
    for i, h in enumerate(header, 1):
        if h and "kks" in str(h).lower() and "модели" in str(h).lower():
            kks_col = i
            break
    if kks_col is None:
        raise RuntimeError("Модели.xlsx: не найден столбец 'KKS модели'")

    merged = list(ws.merged_cells.ranges)
    last_row = find_last_data_row(ws)
    max_col = ws.max_column

    # распространяем объединения
    rows = {}
    for r in range(1, last_row + 1):
        rows[r] = [ws.cell(row=r, column=c).value for c in range(1, max_col + 1)]
    for mr in merged:
        val = ws.cell(row=mr.min_row, column=mr.min_col).value
        for r in range(mr.min_row, mr.max_row + 1):
            for c in range(mr.min_col, mr.max_col + 1):
                if r in rows and c - 1 < len(rows[r]):
                    rows[r][c - 1] = val

    vmerge_cols = set()
    for mr in merged:
        if mr.max_row > mr.min_row:
            for c in range(mr.min_col, mr.max_col + 1):
                vmerge_cols.add(c)

    # фильтрация
    kept, removed = [], 0
    for r in range(2, last_row + 1):
        vals = rows[r]
        kks = vals[kks_col - 1]
        kks_s = str(kks).strip() if kks is not None else ""
        if kks_s and kks_s in exclude:
            removed += 1
            continue
        kept.append(vals)
    log(f"Модели.xlsx: удалено строк — {removed}, осталось — {len(kept)}")

    # пересборка листа
    for mr in merged:
        try:
            ws.unmerge_cells(str(mr))
        except Exception:
            pass
    if ws.max_row > 1:
        ws.delete_rows(2, ws.max_row - 1)
    for i, vals in enumerate(kept, start=2):
        for c_idx, v in enumerate(vals, start=1):
            ws.cell(row=i, column=c_idx, value=v)

        # --- столбцы-исключения и дополнительные столбцы ---
    def find_col(*parts):
        for i, h in enumerate(header, 1):
            if h and all(p in str(h).lower() for p in parts):
                return i
        return None

    type_col    = find_col("тип", "модели")
    signals_col = find_col("отслеживаемые", "сигнал")

        # какие столбцы объединяем по блокам KKS
    merge_cols = set(vmerge_cols) | {kks_col}
    if signals_col:
        merge_cols.add(signals_col)
    if type_col:
        merge_cols.add(type_col)   # Тип модели — тоже по блокам KKS

    # --- блоки по KKS (подряд идущие строки с одним KKS) ---
    blocks = []
    start = 0
    for i in range(1, len(kept) + 1):
        if i == len(kept) or kept[i][kks_col - 1] != kept[start][kks_col - 1]:
            blocks.append((start + 2, i + 1))   # строки листа (с 2-й)
            start = i

    # у столбцов, которые объединяем, значение берём из первой строки блока
    for (r1, r2) in blocks:
        if r2 <= r1:
            continue
        for c in merge_cols:
            try:
                ws.merge_cells(start_row=r1, start_column=c,
                               end_row=r2, end_column=c)
            except Exception:
                pass

    # столбец "#" первым, нумерация по блокам KKS
    add_index_column(ws, ref_col=kks_col)
    format_sheet(ws)


    wb.save(path)


# ---------- 3-4. Параметры.xlsx ----------
def update_params(path, exclude):
    wb = load_workbook(path)

    for sheet_name in ("Параметры", "Правила", "Константы"):
        if sheet_name not in wb.sheetnames:
            continue
        ws = wb[sheet_name]
        header = [c.value for c in ws[1]]
        col = None
        for i, h in enumerate(header, 1):
            if h and "используется" in str(h).lower():
                col = i
                break
        if col is None:
            log(f"Параметры.xlsx[{sheet_name}]: столбец 'Используется в моделях' не найден")
            continue

        last_row = find_last_data_row(ws)
        to_delete, to_update = [], []
        for r in range(2, last_row + 1):
            v = ws.cell(row=r, column=col).value
            if v is None:
                continue
            parts = [p.strip() for p in
                     str(v).replace("\r\n", "\n").replace("\r", "\n").split(";")]
            parts = [p for p in parts if p]
            filtered = [p for p in parts if p not in exclude]
            if not filtered:
                to_delete.append(r)
            elif len(filtered) != len(parts):
                to_update.append((r, ";\n".join(filtered)))

        for r, nv in to_update:
            ws.cell(row=r, column=col, value=nv)
        for r in sorted(to_delete, reverse=True):
            ws.delete_rows(r, 1)
        log(f"Параметры.xlsx[{sheet_name}]: удалено строк — {len(to_delete)}, "
            f"изменено ячеек — {len(to_update)}")

    # п.4 — обрезка Описания в Правилах
    if "Правила" in wb.sheetnames:
        ws = wb["Правила"]
        header = [c.value for c in ws[1]]
        col = None
        for i, h in enumerate(header, 1):
            if h and str(h).strip().lower() == "описание":
                col = i
                break
        if col:
            last_row = find_last_data_row(ws)
            cnt = 0
            for r in range(2, last_row + 1):
                cell = ws.cell(row=r, column=col)
                v = cell.value
                if v is None:
                    continue
                s = str(v).replace("\r\n", "\n").replace("\r", "\n")
                if "\n\n" in s:
                    cell.value = s.split("\n\n")[0]
                    cnt += 1
            log(f"Параметры.xlsx[Правила]: обрезано описаний — {cnt}")

    # столбец "#" первым на каждом листе (без вертикальных объединений)
    for sheet_name in ("Параметры", "Правила", "Константы"):
        if sheet_name in wb.sheetnames:
            add_index_column(wb[sheet_name])
            format_sheet(wb[sheet_name])

    wb.save(path)


# ---------- Карта KKS -> Описание (из уже обрезанного листа Правила) ----------
def load_rules_map(path):
    wb = load_workbook(path, data_only=True)
    if "Правила" not in wb.sheetnames:
        return {}
    ws = wb["Правила"]
    header = [c.value for c in ws[1]]
    kks_col = desc_col = None
    for i, h in enumerate(header, 1):
        if not h:
            continue
        hl = str(h).strip().lower()
        if "kks" in hl and "код" in hl:
            kks_col = i
        elif hl == "описание":
            desc_col = i
    if not kks_col or not desc_col:
        return {}
    last_row = find_last_data_row(ws)
    m = {}
    for r in range(2, last_row + 1):
        kks = ws.cell(row=r, column=kks_col).value
        desc = ws.cell(row=r, column=desc_col).value
        if kks is not None:
            m[str(kks).strip()] = desc
    return m


# ---------- 5. Удалить папки ----------
def delete_folders(models_dir, exclude):
    if not os.path.isdir(models_dir):
        return
    deleted = []
    for name in os.listdir(models_dir):
        p = os.path.join(models_dir, name)
        if os.path.isdir(p) and name.strip() in exclude:
            shutil.rmtree(p)
            deleted.append(name)
    log(f"models/: удалено папок — {len(deleted)}")


# ---------- 6-7. config.json и _meta.json ----------
def update_model_files(models_dir, rules_map):
    n_cfg = n_meta = 0
    for name in sorted(os.listdir(models_dir)):
        folder = os.path.join(models_dir, name)
        if not os.path.isdir(folder):
            continue

        # ---- п.6: config.json ----
        cfg_path = os.path.join(folder, f"{name}.config.json")
        if os.path.isfile(cfg_path):
            try:
                with open(cfg_path, "r", encoding="utf-8") as f:
                    cfg = json.load(f)

                def _update_rule(d):
                    """Обновляет d['description'] по d['kks']. Возвращает True, если ок."""
                    if not isinstance(d, dict):
                        return False
                    kks_cfg = d.get("kks")
                    if kks_cfg is None:
                        log(f"  WARN {name}: diagnostic_rule.kks отсутствует")
                        return False
                    kks_key = str(kks_cfg).strip()
                    new_desc = rules_map.get(kks_key)
                    if new_desc is None:
                        log(f"  WARN {name}: kks '{kks_key}' не найден в Правилах")
                        return False
                    d["description"] = new_desc
                    return True

                dr = cfg.get("diagnostic_rule")
                ok = False
                if isinstance(dr, dict):
                    ok = _update_rule(dr)
                elif isinstance(dr, list):
                    for item in dr:
                        ok = _update_rule(item) or ok
                else:
                    log(f"  WARN {name}: diagnostic_rule не dict/list")

                with open(cfg_path, "w", encoding="utf-8") as f:
                    json.dump(cfg, f, ensure_ascii=False, indent=2)
                if ok:
                    n_cfg += 1
            except Exception as e:
                log(f"  ERROR {cfg_path}: {e}")

        # ---- п.7: _meta.json ----
        meta_path = os.path.join(folder, f"{name}_meta.json")
        if os.path.isfile(meta_path):
            try:
                with open(meta_path, "r", encoding="utf-8") as f:
                    meta = json.load(f)
                for k in ("input_hashes", "train_hash", "trained_at", "job_id"):
                    meta.pop(k, None)
                with open(meta_path, "w", encoding="utf-8") as f:
                    json.dump(meta, f, ensure_ascii=False, indent=2)
                n_meta += 1
            except Exception as e:
                log(f"  ERROR {meta_path}: {e}")

    log(f"Обновлено config.json — {n_cfg}, _meta.json — {n_meta}")


# ---------- 8. Копирование ----------
def aggregate_copy(models_dir, configs_dir, nn_dir):
    for d in (configs_dir, nn_dir):
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d)

    n_cfg = n_k = n_m = 0
    for name in sorted(os.listdir(models_dir)):
        folder = os.path.join(models_dir, name)
        if not os.path.isdir(folder):
            continue
        for fname, dst_dir, kind in (
            (f"{name}.config.json", configs_dir, "cfg"),
            (f"{name}.keras",       nn_dir,      "keras"),
            (f"{name}_meta.json",   nn_dir,      "meta"),
        ):
            src = os.path.join(folder, fname)
            if os.path.isfile(src):
                shutil.copy2(src, dst_dir)
                if kind == "cfg":
                    n_cfg += 1
                elif kind == "keras":
                    n_k += 1
                else:
                    n_m += 1
    log(f"Скопировано: config — {n_cfg}, keras — {n_k}, meta — {n_m}")


def main():
    if not os.path.isdir(ROOT):
        sys.exit(f"Нет корневой папки: {ROOT}")

    exclude = load_exclude_list(EXCLUDE_FILE)
    log(f"Моделей к исключению: {len(exclude)}: {sorted(exclude)}\n")

    update_models(MODELS_XLSX, exclude)
    update_params(PARAMS_XLSX, exclude)
    delete_folders(MODELS_DIR, exclude)

    rules_map = load_rules_map(PARAMS_XLSX)
    log(f"Правил загружено: {len(rules_map)}")
    update_model_files(MODELS_DIR, rules_map)

    aggregate_copy(MODELS_DIR, CONFIGS_DIR, NN_DIR)
    log("\nГотово.")


if __name__ == "__main__":
    main()

Моделей к исключению: 26: ['10MAG10FP_MAX_NN', '10MKD11CY41_42_MAX_NN', '10MKD12CY41_42_MAX_NN', '11MBA11CP101_NN', '11MBA11CT901_NN', '11MBD11CY940_NN', '11MBD12CY940_NN', '11MBM12CT_13_18_NN', '11MBM12CT_19_24_NN', '11MBM12CT_1_6_NN', '11MBM12CT_7_12_NN', '11MBM12CT_AVG_NN', '11MBY01EZ151_NN', '11MBY01EZ152_NN', '11MBY01EZ153_NN', '11MKD11CY901_NN', '11MKD12CY901_NN', '1MKC10CT183A_NN', 'L0HAD10CL901_NN', 'L0HAD40CL901_NN', 'L0HAD70CL901_NN', 'L0HNE01CQ003_NN', 'L0HNE01CQ004_2_NN', 'L0HNE01CQ004_NN', 'L0LAA01CL901_NN', 'L0NDA40FT_MED_NN']

Модели.xlsx: удалено строк — 124, осталось — 298
Параметры.xlsx[Параметры]: удалено строк — 54, изменено ячеек — 8
Параметры.xlsx[Правила]: удалено строк — 21, изменено ячеек — 0
Параметры.xlsx[Константы]: удалено строк — 70, изменено ячеек — 2
Параметры.xlsx[Правила]: обрезано описаний — 58
models/: удалено папок — 26
Правил загружено: 59
Обновлено config.json — 59, _meta.json — 59
Скопировано: config — 59, keras — 59, meta — 59

Готово.


In [4]:
import pandas as pd

df1 = pd.read_excel('/home/tishchenkova@lofi.pgt/Документы/Договора/ГЭХ/2026.11.23/Сводки моделей_актуальные.xlsx')
col1 = set(df1['KKS модели'].unique())
df2 = pd.read_excel('/home/tishchenkova@lofi.pgt/Документы/Договора/ГЭХ/2026.11.23/Модели.xlsx')
col2 = set(df2['KKS модели'].unique())
col1-col2


{'L0NDA40CT001_NN'}